# batss CRN benchmarks

Each CRN below defines a `CRNSpec` plus a handful of per-CRN variables (`benchmark_end_time`, `trajectory_n`, `trajectory_end_time`). To benchmark and plot a CRN:

1. Run its definition cell.
2. Run the **Run** cell at the bottom.

Runtime measurements are cached to JSON under `CACHE_DIR`, so rerunning the **Run** cell skips already-computed `(backend, n)` pairs.

In [3]:
import batss as bt
from batss.benchmarking import CRNSpec, benchmark_runtimes, plot_runtimes, plot_trajectory

DATA_DIR = "data"

## Dimerization

$M + M \rightleftharpoons D$, reversible with both rates 1.

In [4]:
m, d = bt.species("M D")
spec = CRNSpec(
    name="dimerization",
    rxns=[
        (m + m >> d).k(1),
        (d >> m + m).k(1),
    ],
    inits_from_n=lambda n: {m: n},
)
benchmark_end_time = 0.5
trajectory_n = 10**6
trajectory_end_time = 2.0

## Lotka–Volterra

$R + F \to 2F$, $R \to 2R$, $F \to \varnothing$, all at rate 1. Half-and-half initial split.

In [ ]:
r, f = bt.species("R F")
spec = CRNSpec(
    name="lotka_volterra",
    rxns=[
        (r + f >> 2 * f).k(1),
        (r >> 2 * r).k(1),
        (f >> None).k(1),
    ],
    inits_from_n=lambda n: {r: n // 2, f: n - n // 2},
)
benchmark_end_time = 1.0
trajectory_n = 10**5
trajectory_end_time = 20.0

## Rössler

Three-species chaotic oscillator; rates as in `rossler_plotting.py`. Thirds initial split.

In [ ]:
x1, x2, x3 = bt.species("X1 X2 X3")
spec = CRNSpec(
    name="rossler",
    rxns=[
        (x1 >> 2 * x1).k(30),
        (2 * x1 >> x1).k(0.5),
        (x2 + x1 >> 2 * x2).k(1),
        (x2 >> None).k(10),
        (x1 + x3 >> None).k(1),
        (x3 >> 2 * x3).k(16.5),
        (2 * x3 >> x3).k(0.5),
    ],
    inits_from_n=lambda n: {x1: n // 3, x2: n // 3, x3: n // 3},
)
benchmark_end_time = 1.0
trajectory_n = 10**4
trajectory_end_time = 8.0

## Run

Runs benchmarks and produces both plots for whichever CRN cell above was run most recently.

In [5]:
pop_sizes = [10**e for e in range(3, 7)]
benchmark_runtimes(
    spec,
    pop_sizes=pop_sizes,
    end_time=benchmark_end_time,
    cache_dir=DATA_DIR,
    overwrite=True,
)
plot_runtimes(
    spec,
    end_time=benchmark_end_time,
    cache_dir=DATA_DIR,
    out_path=f"{DATA_DIR}/{spec.name}_runtime_t{benchmark_end_time}.pdf",
)
plot_trajectory(
    spec,
    n=trajectory_n,
    end_time=trajectory_end_time,
    seed=1,
    num_samples=500,
    out_path=f"{DATA_DIR}/{spec.name}_trajectory_n{trajectory_n}_t{trajectory_end_time}.pdf",
)

benchmarking dimerization on batss -> data\dimerization_runtime_batss_t0.5.json


AttributeError: 'builtins.SimulatorCRNMultiBatch' object has no attribute 'discrete_batched_steps_total'